In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/filtered_complaints.csv"
)

print(df.shape)
df.head()

(353769, 20)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,cleaned_text
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91,a xxxx xxxx card was opened under my name by a...
1,2025-06-13,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I made the mistake of using my wellsfargo debi...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,ID,83815,NaN,Consent provided,Web,2025-06-13,Closed with explanation,Yes,NaN,14061897,109,i made the mistake of using my wellsfargo debi...
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156,dear cfpb i have a secured credit card with ci...
3,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233,i have a citi rewards cards the credit balance...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454,b i am writing to dispute the following charge...


In [2]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID', 'word_count',
       'cleaned_text'],
      dtype='object')

In [3]:
df["Product"].value_counts()

Product
Checking or savings account                                140319
Money transfer, virtual currency, or money service          97188
Credit card                                                 80667
Payday loan, title loan, or personal loan                   17238
Consumer Loan                                                9461
Payday loan, title loan, personal loan, or advance loan      8896
Name: count, dtype: int64

In [4]:
df["Product"].value_counts(normalize=True)

Product
Checking or savings account                                0.396640
Money transfer, virtual currency, or money service         0.274722
Credit card                                                0.228022
Payday loan, title loan, or personal loan                  0.048727
Consumer Loan                                              0.026743
Payday loan, title loan, personal loan, or advance loan    0.025146
Name: proportion, dtype: float64

Creating a Stratified Sample

In [5]:
sample_size = 12000

sample_df = (
    df.groupby("Product", group_keys=False)
      .apply(
          lambda x: x.sample(
              frac=sample_size / len(df),
              random_state=42
          )
      )
)

print(sample_df.shape)

(12001, 20)


/var/folders/43/1hrrs9l51tg3wckl1gxm1mrh0000gn/T/ipykernel_12876/669828748.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Product", group_keys=False)


In [6]:
sample_df["Product"].value_counts(normalize=True)

Product
Checking or savings account                                0.396634
Money transfer, virtual currency, or money service         0.274727
Credit card                                                0.227981
Payday loan, title loan, or personal loan                  0.048746
Consumer Loan                                              0.026748
Payday loan, title loan, personal loan, or advance loan    0.025165
Name: proportion, dtype: float64

Preparing the Text Column

In [7]:
sample_df["cleaned_text"] = (
    sample_df["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
)

Chunking

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [9]:
sample_text = sample_df["cleaned_text"].iloc[0]

chunks = text_splitter.split_text(sample_text)

print(len(chunks))
print(chunks[0][:200])

1
Bank located in has received my check of {$1400.00} showed no proof of fraudulent volunteering closing out my accounts while in Her presenc. Walked out with {$200.00}, instead of {$1600.00}, closed ac


Creating Chunk Dataset

In [10]:
chunk_records = []

for _, row in sample_df.iterrows():

    chunks = text_splitter.split_text(
        row["cleaned_text"]
    )

    for i, chunk in enumerate(chunks):

        chunk_records.append({
            "complaint_id": row["Complaint ID"],
            "product": row["Product"],
            "chunk_index": i,
            "text": chunk
        })

In [11]:
chunk_df = pd.DataFrame(
    chunk_records
)

print(chunk_df.shape)

(38652, 4)


Generate Embeddings

In [12]:
import numpy as np
print(np.__version__)

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("OK")

1.26.4


/Users/new/Desktop/rag-complaint-chatbot/venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/new/Desktop/rag-complaint-chatbot/venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

OK


In [13]:
texts = chunk_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1208 [00:00<?, ?it/s]

In [14]:
embeddings.shape

(38652, 384)

Creating ChromaDB Vector Store

In [16]:
import chromadb

In [17]:
client = chromadb.PersistentClient(
    path="../vector_store/chroma_db"
)

In [18]:
collection = client.get_or_create_collection(
    name="complaints"
)

Store Chunks and Metadata

In [19]:
batch_size = 1000

In [20]:
for start in range(
    0,
    len(chunk_df),
    batch_size
):

    end = start + batch_size

    batch = chunk_df.iloc[start:end]

    collection.add(
        ids=[
            str(i)
            for i in batch.index
        ],

        documents=batch[
            "text"
        ].tolist(),

        embeddings=embeddings[
            start:end
        ].tolist(),

        metadatas=[
            {
                "complaint_id":
                    str(cid),

                "product":
                    product,

                "chunk_index":
                    int(chunk_idx)
            }

            for cid,
                product,
                chunk_idx in zip(
                    batch["complaint_id"],
                    batch["product"],
                    batch["chunk_index"]
                )
        ]
    )

In [21]:
collection.count()

38652

Test Retrieval

In [22]:
query = (
    "customers unhappy with credit cards"
)

In [23]:
query_embedding = model.encode(
    query
)

In [24]:
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],

    n_results=5
)

In [25]:
results["documents"][0]

['credit cards with this company and I have always been happy with them but now I am having very negative feelings towards this company because they are agree that it was not made clear to me what is going on while on the phone and the representatives that I talked to dropped the ball by not telling me, but that it is all my fault and I just have to deal with the consequences now.',
 'I was offered to sign up for a store credit card by an employee. i like to shop but i felt ripped off. because alot of the items i purchase was low in cost. first the cashier told me my payments were only XXXX a month and it would not change and when i look my card was getting hire in cost. well i did make several arrangement that was not kept by the customer service. it starts of good with these cards then they become harrasing charging over the limit fees and late fees and other hidden',
 "my card was declining despite having the funds. Instead they give me run around responses that don't answer my ques

In [27]:
len(embeddings)

38652

In [28]:
chunk_df.head()

,complaint_id,product,chunk_index,text
0,11750738,Checking or savings account,0,Bank located in has received my check of {$140...
1,7166095,Checking or savings account,0,Overdraft fees being charged and ATM fees bein...
2,8225373,Checking or savings account,0,I am writing to formally submit a complaint ag...
3,8225373,Checking or savings account,1,"In the initial dispute ( Claim # XXXX ), Capit..."
4,8225373,Checking or savings account,2,"Regrettably, the situation escalated when I di..."
